# LogiScan Stage 2 — Coarse Classifier Training
## 4-Class Fallacy Category Detection

**Classes:** Formal, Informal (Relevance), Informal (Ambiguity), Informal (Presumption)
**Model:** DistilBERT (lightweight, routes to Stage 3 + Z3)
**Output:** `stage2_coarse_classifier.zip`
**Estimated time:** 10-15 minutes on T4x2 GPU (Kaggle)

In [ ]:
# 1. Install
!pip install -q transformers torch scikit-learn tqdm

In [ ]:
# 2. Imports
import json
from collections import Counter
from pathlib import Path

import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}, {torch.cuda.device_count()} device(s) else 'CPU'}")

### Upload stage2_coarse_data.json

In [ ]:
# 3. Load data from Kaggle datasetDATA_PATH = "/kaggle/input/logiscan-unified-data/unified_training_data.json"with open(DATA_PATH) as f:
    raw = json.load(f)

# 1. Broad Category Mapping (Taxonomy 1.2.0)
COARSE_MAP = {
    "ad_hominem": "Informal (Relevance)",
    "affirming_consequent": "Formal",
    "appeal_to_authority": "Informal (Relevance)",
    "appeal_to_emotion": "Informal (Relevance)",
    "appeal_to_nature": "Informal (Relevance)",
    "appeal_to_tradition": "Informal (Relevance)",
    "bandwagon": "Informal (Relevance)",
    "begging_the_question": "Informal (Presumption)",
    "composition": "Informal (Presumption)",
    "denying_antecedent": "Formal",
    "division": "Informal (Presumption)",
    "equivocation": "Informal (Ambiguity)",
    "false_cause": "Informal (Presumption)",
    "false_dilemma": "Informal (Presumption)",
    "hasty_generalization": "Informal (Presumption)",
    "moving_goalposts": "Informal (Relevance)",
    "no_true_scotsman": "Informal (Ambiguity)",
    "red_herring": "Informal (Relevance)",
    "slippery_slope": "Informal (Presumption)",
    "straw_man": "Informal (Relevance)",
    "tu_quoque": "Informal (Relevance)",
    "tu_quoque_contextual": "Informal (Relevance)",
    "factual_statement": "Non-Fallacious",
    "valid_reasoning": "Non-Fallacious"
}

texts, labels_raw = [], []
for d in raw:
    texts.append(d["text"])
    labels_raw.append(COARSE_MAP.get(d["fallacy"], "Informal (Other)"))

categories = sorted(list(set(labels_raw)))
label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for c, i in label2id.items()}

labels = [label2id[l] for l in labels_raw]

print(f"Total Samples: {len(texts)}")
print(f"Categories: {categories}")

In [ ]:
# 4. Split
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.20, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
# 5. Dataset with class balancing
class CoarseDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Balance classes
class_counts = Counter(y_train)
weights = {c: 1.0/max(count, 1) for c, count in class_counts.items()}
sample_weights = [weights[l] for l in y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = CoarseDataset(X_train, y_train, tokenizer)
val_ds = CoarseDataset(X_val, y_val, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=64)
print(f"Batches — Train: {len(train_loader)}, Val: {len(val_loader)}")

In [ ]:
# 6. Baseline
majority = Counter(y_val).most_common(1)[0][1] / len(y_val)
print(f"Majority baseline: {majority:.4f}")

In [ ]:
# 7. Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print(f"Epochs: {epochs}")

In [ ]:
# 8. Train
best_f1 = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.3f}"})

    # Validate
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            p = torch.argmax(out.logits, dim=1).cpu().numpy()
            preds.extend(p)
            truths.extend(batch["label"].numpy())

    acc = accuracy_score(truths, preds)
    f1 = f1_score(truths, preds, average="macro")
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, acc={acc:.4f}, macro_f1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        Path("stage2_coarse_classifier").mkdir(exist_ok=True)
        model.save_pretrained("stage2_coarse_classifier")
        tokenizer.save_pretrained("stage2_coarse_classifier")
        print(f"  ✅ Saved (f1={f1:.4f})")

print(f"\nBest macro F1: {best_f1:.4f}")

In [ ]:
# 9. Evaluation
print(classification_report(truths, preds, target_names=categories, zero_division=0))

# Quick test
tests = [
    ("If it rains, the ground is wet. The ground is wet, therefore it rained.", "Formal"),
    ("You cannot trust him because he is not a scientist.", "Informal (Relevance)"),
    ("A feather is light. What is light cannot be dark. So a feather cannot be dark.", "Informal (Ambiguity)"),
    ("Obviously this is the right choice because it's common sense.", "Informal (Presumption)"),
]

model.eval()
print("\nTest predictions:")
for text, expected in tests:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1)[0]
        pred_idx = torch.argmax(probs).item()
        pred_label = id2label[pred_idx]
        conf = probs[pred_idx].item()
    match = "✅" if pred_label == expected else "❌"
    print(f"  {match} Expected: {expected:30s} → Predicted: {pred_label:30s} ({conf:.2f})")
    print(f"     Text: {text[:80]}...")
    print()

In [ ]:
# 10. Save output to Kaggle working directory!zip -r stage2_coarse_classifier.zip stage2_coarse_classifier/print(f"\n✅ Model saved to /kaggle/working/stage2_coarse_classifier.zip")print("\nTo download: commit the notebook, then download from the output tab.")print("\nOn your machine:")print("  unzip stage2_coarse_classifier.zip -d models/")